### Day 4 — Feature Engineering & Hyperparameter Tuning  

In [19]:
import pandas as pd

In [20]:
df = pd.read_csv('Housing Prices/housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [21]:
df.isnull().sum() # total_bedrooms is null 207

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [22]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median()) # chosen median because it is not affected by outliers

In [23]:
df.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='str')

In [24]:
# 1. Number of rooms per household
df['rooms_per_household'] = df['total_rooms'] / df['households']

# 2.Number of bedrooms per room (ratio of rooms allocated for sleeping)
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']

# 3. Number of people per household (average household size)
df['population_per_household'] = df['population'] / df['households']

df[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].head()

,rooms_per_household,bedrooms_per_room,population_per_household
0,6.984127,0.146591,2.555556
1,6.238137,0.155797,2.109842
2,8.288136,0.129516,2.802260
3,5.817352,0.184458,2.547945
4,6.281853,0.172096,2.181467


**1. `rooms_per_household` = total_rooms / households** — normalizes total 
room count by household count, giving a per-home size measure rather than 
a raw total confounded by how many households live in the block.

**2. `bedrooms_per_room` = total_bedrooms / total_rooms** — captures the 
proportion of rooms that are bedrooms; a lower ratio suggests larger, more 
spacious homes, which tends to correlate with higher property value.

**3. `population_per_household` = population / households** — estimates 
average household size; unusually high values may indicate overcrowding, 
often associated with lower-income areas and lower prices.

These ratio-based features isolate per-unit signal that raw totals hide, 
since raw totals are confounded by the number of households in a block.

In [25]:
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 1.0]
}

rf_model = RandomForestRegressor(random_state=42)

`n_estimators` and `max_depth` control model complexity and overfitting 
risk; `min_samples_split`/`min_samples_leaf` regularize by limiting how 
finely the model fits small subsets; `max_features` controls per-split 
randomness, key to Random Forest's variance reduction. Use with 
`RandomizedSearchCV` (324 combinations) and a regression scorer like 
`neg_root_mean_squared_error`, since the target is continuous.

In [26]:
df.drop('ocean_proximity', axis=1, inplace=True)
df_sample = df.sample(n=2000, random_state=42)
y = df_sample['median_house_value']
X = df_sample.drop('median_house_value', axis=1)

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


KeyboardInterrupt: 

Since the grid contains 324 combinations x 5 folds = 1,620 training operations, this can take a very long time (minutes to hours depending on the data size and the computer's processing power). To speed up the process: I use `RandomizedSearchCV` (Tries a random sample instead of all combinations)

In [29]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_grid,
    n_iter=30,          #Try 30 random combinations instead of 324
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best CV RMSE:", -random_search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 20}
Best CV RMSE: 61895.54919717139


#### RandomizedSearchCV Results

**Best Parameters:** `{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 20}`
**Best CV RMSE:** `61895.55`

The best parameter combination was selected based on the lowest mean RMSE 
across 5 cross-validation folds. Compared to the baseline model 
(`n_estimators=100`, default depth), this represents no meaningful 
improvement — in fact a ~23.3% *increase* in RMSE (from 50199.09 to 
61895.55) — suggesting the default parameters performed better on this 
dataset than the tuned combination, or that the two scores are not 
directly comparable.

In [30]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

# model from week 3 
baseline_rf = RandomForestRegressor(n_estimators=100, random_state=42)

baseline_cv_scores = cross_val_score(
    baseline_rf, X_train, y_train,
    cv=5,
    scoring='neg_root_mean_squared_error'
)
baseline_rmse = -baseline_cv_scores.mean()

print(f"Baseline (untuned) CV RMSE: {baseline_rmse:.2f}")
print(f"Tuned CV RMSE: {-random_search.best_score_:.2f}")
print(f"Improvement: {baseline_rmse - (-random_search.best_score_):.2f}")
print(f"Improvement %: {(baseline_rmse - (-random_search.best_score_)) / baseline_rmse * 100:.2f}%")

Baseline (untuned) CV RMSE: 62180.64
Tuned CV RMSE: 61895.55
Improvement: 285.09
Improvement %: 0.46%


The best parameters were selected based on the lowest mean RMSE across 5 
cross-validation folds, compared against the baseline using the same CV 
methodology.

This represents no meaningful improvement (~0.46%, within normal CV 
noise), suggesting the default parameters were already close to optimal 
for this dataset. Random Forest is generally robust to hyperparameter 
changes within reasonable ranges, so further gains are more likely to 
come from feature engineering or a different model class (e.g., 
Gradient Boosting) than continued RF tuning.

In [31]:
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': random_search.best_estimator_.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(10))

                     feature  importance
7              median_income    0.520170
10  population_per_household    0.141715
1                   latitude    0.072353
0                  longitude    0.068930
2         housing_median_age    0.057803
8        rooms_per_household    0.046101
9          bedrooms_per_room    0.025000
6                 households    0.018879
5                 population    0.017297
4             total_bedrooms    0.017188


#### Feature Importance

| Feature | Importance |
|---|---|
| median_income | 0.520 |
| population_per_household | 0.142 |
| latitude | 0.072 |
| longitude | 0.069 |
| housing_median_age | 0.058 |
| rooms_per_household | 0.046 |
| bedrooms_per_room | 0.025 |
| households | 0.019 |
| population | 0.017 |
| total_bedrooms | 0.017 |

**Most important feature:** `median_income`, accounting for ~52% of total 
importance — by far the strongest predictor of `median_house_value`. The 
engineered feature `population_per_household` ranks second (~14%), 
outperforming several original features (`households`, `population`, 
`total_bedrooms`), which confirms it added meaningful signal beyond the 
raw variables it was derived from.

In [33]:
results_df = pd.DataFrame(random_search.cv_results_)

for param in ['param_n_estimators', 'param_max_depth', 'param_min_samples_leaf', 
              'param_min_samples_split', 'param_max_features']:
    print(f"\n{param}:")
    print(results_df.groupby(param)['mean_test_score'].mean().sort_values(ascending=False))


param_n_estimators:
param_n_estimators
200   -63081.379674
300   -63106.786037
100   -63431.033990
Name: mean_test_score, dtype: float64

param_max_depth:
param_max_depth
30   -62819.297406
20   -63169.401490
10   -63586.571399
Name: mean_test_score, dtype: float64

param_min_samples_leaf:
param_min_samples_leaf
2   -62839.493063
1   -62965.318142
4   -63681.043784
Name: mean_test_score, dtype: float64

param_min_samples_split:
param_min_samples_split
5    -62825.352976
2    -63103.817208
10   -63641.455680
Name: mean_test_score, dtype: float64

param_max_features:
param_max_features
1.0    -62340.487793
sqrt   -63526.929501
log2   -63786.747840
Name: mean_test_score, dtype: float64


#### Hyperparameter Sensitivity

| Hyperparameter | Value | Mean CV RMSE | Range Across Values |
|---|---|---|---|
| max_features | 1.0 / sqrt / log2 | 62,340.49 / 63,526.93 / 63,786.75 | **1,446.26** |
| min_samples_leaf | 2 / 1 / 4 | 62,839.49 / 62,965.32 / 63,681.04 | 841.55 |
| min_samples_split | 5 / 2 / 10 | 62,825.35 / 63,103.82 / 63,641.46 | 816.11 |
| max_depth | 30 / 20 / 10 | 62,819.30 / 63,169.40 / 63,586.57 | 767.27 |
| n_estimators | 200 / 300 / 100 | 63,081.38 / 63,106.79 / 63,431.03 | 349.65 |

**Most impactful hyperparameter:** `max_features`, with the widest RMSE 
spread (~1,446) across its tested values. Using `max_features=1.0` 
(all features considered at each split) clearly outperformed both 
`sqrt` and `log2`, suggesting the dataset's small feature set benefits 
from less restriction on split candidates. `n_estimators` had the 
smallest effect, indicating the ensemble had already converged by 100–200 
trees.